In [ ]:
# ════════════════════════════════════════════════════════════════════
# Bootstrap — clone the Lamahat repo (single source of truth).
# Code, fonts/ and resources/ arrive together at ONE commit, so this
# notebook and the Streamlit app can never drift apart.  Pin BRANCH to
# a feature branch to test unreleased work; leave "main" for releases.
#
# Pinned to the P7 movie-quality branch: main does NOT yet carry the
# caption-synchronisation work (P7.15–P7.17).  Running this notebook
# against main reproduces the drift that starts around 1:13.  Put
# BRANCH back to "main" once that branch is merged.
# ════════════════════════════════════════════════════════════════════
REPO   = "https://github.com/abdoljh/Lamahat.git"
BRANCH = "claude/phase3-movie-quality-d1n58l"

import os, shutil
if os.path.isdir("/content/Lamahat"):
    shutil.rmtree("/content/Lamahat")
!git clone --depth 1 --branch {BRANCH} {REPO} /content/Lamahat
%cd /content/Lamahat
!git log -1 --pretty="✅ Running at commit: %h  %s"


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Settings — edit before running.
# ════════════════════════════════════════════════════════════════════
SOURCE = "drive"           # "zip" (upload plan + review/) or "drive"
DRIVE_SOURCE_DIR = "/content/drive/MyDrive/_Phase3/sources"
ARCHIVE_ZIP_NAME = "Archive.zip"

# ── Book Title & Main Character  ────────────────────────────────────
BOOK_TITLE     = "مذكرات جعفر العسكري"
CHARACTER_NAME = "Jafar al-Askari"

# ── Look ────────────────────────────────────────────────────────────
BOOK_COVER_PICK   = 2          # 1..N over resources/book_cover/
BOOK_COVER_FIT    = "contain"  # fill | contain | blur_pad
BOOK_COVER_ALIGN  = "left"    # center | left | right
TYPOGRAPHY_FAMILY = "B"        # A | B | C
GRADE             = "warm"  # warm | cool | neutral | bw
CAPTION_BACKPLATE = "off"      # off | subtle | solid
TEXT_SCRIM        = "auto"     # auto | off | soft | band  (auto = plate only on bright/busy frames)
OVERLAY_ANCHOR    = "auto"     # auto | center | lower  (auto = quotes/names lower-third, section marks centered)
TITLE_SUBTITLE    = "١٨٨٥ - ١٩٣٦"         # optional sub-line under the main title (author / dates); "" = title only
WORD_REVEAL       = True      # word-by-word reveal on over-image quotes (experimental)
PHOTO_BANK_MAX_USES = 1      # how many shots one curated photo may cover (2 spreads the bank further)
GRADE_MAP         = ""         # optional per-section grading JSON path, e.g. "resources/grade_map.json"
                               #   {"opening":"neutral","point_2":"cool","closing":"warm"} — unmapped → GRADE
MUSIC_DB          = -12.0      # music bed level in dB (default -18)

# ── Text styling (blank = pipeline default) ─────────────────────────
TITLE_SIZE    = 1.0   # main-title size multiplier (1.2 = 20% larger)
TITLE_COLOR   = ""    # "#RRGGBB" or "" -> family default (aged gold)
CAPTION_SIZE  = 1.5   # caption size multiplier
CAPTION_COLOR = ""    # "#RRGGBB" or "" -> white
CAPTION_POS   = ""    # fraction of height from bottom, e.g. "0.08"; "" -> default
# Captions are burned ON and these knobs are wired into the render cell
# (they were being defined here and silently dropped before).

# ── Output Files & Directories ──────────────────────────────────────
OUTPUT_BASE_DIR         = "output"
OUTPUT_FILE             = f"{OUTPUT_BASE_DIR}/final_cut_{TYPOGRAPHY_FAMILY}.mp4"
LOG_FILE                = f"{OUTPUT_BASE_DIR}/render.log"
CONDITIONED_ZIP_FILE    = f"{OUTPUT_BASE_DIR}/conditioned.zip"
FINAL_ZIP_FILE          = "output_files.zip"
RO_ZIP_FILE             = "output_files_ro.zip"
DRIVE_SAVE_DIR          = "/content/drive/MyDrive/_Phase3/output"
DRIVE_SAVE_RO_DIR       = "/content/drive/MyDrive/_Phase3/output/ro"

# Generated artifacts — supplied at render time (upload .zip or Drive):
PLAN_FILE    = f"{OUTPUT_BASE_DIR}/shot_plan.json"
TIMINGS_FILE = f"{OUTPUT_BASE_DIR}/word_timings.json"
REVIEW_DIR   = f"{OUTPUT_BASE_DIR}/review"
REVIEW_FILE  = f"{OUTPUT_BASE_DIR}/review.zip"

# ── Quality gates ───────────────────────────────────────────────────
# audit_captions.py fails outright on any DUPLICATED / LEAKED /
# CAPTION-OVER-CARD caption — those are the defects the screenings kept
# catching.  A few LOST words are the current floor on a real plan (10
# on the last verified run), so they get a budget instead.  Raise this
# only after reading the report; a jump means a regression, not noise.
MAX_LOST_WORDS = 12

# Committed inputs
SCRIPT_FILE = "resources/script/main_script.txt"
AUDIO_FILE  = "resources/audio/narration.mp3"
MUSIC_BED   = "resources/audio/bg_music.mp3"

# print(f"SOURCE={SOURCE!r}  OUTPUT={OUTPUT_FILE!r}  GRADE={GRADE!r}  SCRIM={TEXT_SCRIM!r}")
print(f"OUTPUT= {OUTPUT_FILE}\nGRADE = {GRADE}\nSCRIM = {TEXT_SCRIM}\n")
# ── Optional: host big/private assets on Drive instead of the repo ──
# (pools: character/, book_cover/, photo_bank/ are discovered there)
# import os; os.environ["LAMAHAT_RESOURCES"] = "/content/drive/MyDrive/Lamahat/resources"
# ── Optional: persist the image + vision-score cache on Drive ──
# (survives the Colab VM, so re-running prebuild costs ~nothing)
# import os; os.environ["LAMAHAT_CACHE"] = "/content/drive/MyDrive/Lamahat/cache"


In [3]:
# Colab-only dependencies (whisperx, openai-whisper, anthropic, Arabic
# shaping).  Streamlit Cloud installs requirements.txt; this file adds
# only the Colab extras on top of Colab's preinstalled stack.
!pip install -q -r requirements-colab.txt
print("✅ Colab dependencies installed!")
!ffmpeg -version 2>&1 | head -1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 124.9 MB/s e

In [4]:
# OPTIONAL
# Check resources availability
from pathlib import Path
import os

resources_path = Path("resources").resolve()  # repo clone cwd

character_path = resources_path / "character"
book_cover_path = resources_path / "book_cover"

# Display contents of resources/character
print(f"Contents of {character_path}:")
if character_path.exists() and character_path.is_dir():
    chars = sorted(os.listdir(character_path))
    print(f"  Number of items: {len(chars)}")
    for item in chars:
        print(f"    - {item}")
else:
    print("  Directory not found or is not a directory.")

print("\n")

# Display contents of resources/book_cover
print(f"Contents of {book_cover_path}:")
if book_cover_path.exists() and book_cover_path.is_dir():
    covers = sorted(os.listdir(book_cover_path))
    print(f"  Number of items: {len(covers)}")
    for item in covers:
        print(f"    - {item}")
else:
    print("  Directory not found or is not a directory.")


Contents of /content/Lamahat/resources/character:
  Number of items: 11
    - alayubi_jafar_alsaeed.jpg
    - jafar_at_window_w.jpg
    - jafar_in_cairo_conf_1921.jpg
    - jafar_in_office.jpg
    - jafar_in_quweira.jpg
    - jafar_in_uniform.jpg
    - jafar_order_of_the_medjidie.jpg
    - jafar_profile.jpg
    - memoirs_3dw1.jpg
    - memoirs_3dw2.jpg
    - memoirs_in_shelf.jpg


Contents of /content/Lamahat/resources/book_cover:
  Number of items: 4
    - memoirs_3d.jpeg
    - memoirs_cover.jpg
    - memoirs_opened.jpg
    - memoirs_vertical.jpg


In [5]:
# OPTIONAL
# Smoke rendering
from pathlib import Path
from phase3.typography_common import TypographySpec
from phase3.typography import render
import os

# Ensure the target directory exists
output_dir = '/content/temp'
os.makedirs(output_dir, exist_ok=True)

rendered_files_expected = []
for fam in ("A", "B", "C"):
    for tpl in ("title_card", "section_mark", "pull_quote", "name_reveal", "date_stamp"):
        spec = TypographySpec(
            template=tpl, family=fam,
            #text="مذكرات جعفر العسكري",
            text = "مُذَكِّراتُ جَعْفَرِ العَسْكَرِيِّ",
            subtitle="١٨٨٥ - ١٩٣٦",
            width=1920, height=1080,
        )
        file_path = Path(f"{output_dir}/typo_{fam}_{tpl}.png")
        render(spec, file_path)
        rendered_files_expected.append(file_path)

print("💨 Typography examples rendering process completed.")

# Verification step
print(f"\nVerifying files in '{output_dir}':")
found_files_actual = []
for f_expected in rendered_files_expected:
    if f_expected.exists():
        found_files_actual.append(f_expected.name)
        print(f"  ✅ Found: {f_expected.name}")
    else:
        print(f"  ❌ NOT found: {f_expected.name}")

if not found_files_actual:
    print(f"No rendered image files were found in '{output_dir}'.")
    print("This suggests an issue with the `render` function from `phase3.typography` not creating the files as expected, or writing them to a different location.")
else:
    print(f"\nSuccessfully found {len(found_files_actual)} rendered files.")


💨 Typography examples rendering process completed.

Verifying files in '/content/temp':
  ✅ Found: typo_A_title_card.png
  ✅ Found: typo_A_section_mark.png
  ✅ Found: typo_A_pull_quote.png
  ✅ Found: typo_A_name_reveal.png
  ✅ Found: typo_A_date_stamp.png
  ✅ Found: typo_B_title_card.png
  ✅ Found: typo_B_section_mark.png
  ✅ Found: typo_B_pull_quote.png
  ✅ Found: typo_B_name_reveal.png
  ✅ Found: typo_B_date_stamp.png
  ✅ Found: typo_C_title_card.png
  ✅ Found: typo_C_section_mark.png
  ✅ Found: typo_C_pull_quote.png
  ✅ Found: typo_C_name_reveal.png
  ✅ Found: typo_C_date_stamp.png

Successfully found 15 rendered files.


In [6]:
# OPTIONAL
# Zip typography images
import os
import zipfile

image_dir = '/content/temp'
zip_filename = 'typography_images.zip'

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    if os.path.exists(image_dir):
        for root, dirs, files in os.walk(image_dir):
            for file in files:
                if file.endswith('.png'): # Only zip PNG images
                    file_path = os.path.join(root, file)
                    # Add file to zip, preserving directory structure relative to 'image_dir'
                    zipf.write(file_path, os.path.relpath(file_path, image_dir))
                    files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing:")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print(f"  (No .png files found in '{image_dir}' to zip.)")

🤐 Successfully created 'typography_images.zip' containing:
  - /content/temp/typo_C_pull_quote.png
  - /content/temp/typo_C_section_mark.png
  - /content/temp/typo_B_title_card.png
  - /content/temp/typo_B_pull_quote.png
  - /content/temp/typo_B_name_reveal.png
  - /content/temp/typo_C_name_reveal.png
  - /content/temp/typo_A_section_mark.png
  - /content/temp/typo_B_section_mark.png
  - /content/temp/typo_A_date_stamp.png
  - /content/temp/typo_C_title_card.png
  - /content/temp/typo_C_date_stamp.png
  - /content/temp/typo_A_title_card.png
  - /content/temp/typo_A_name_reveal.png
  - /content/temp/typo_B_date_stamp.png
  - /content/temp/typo_A_pull_quote.png


In [7]:
# render_plan help (OPTIONAL)
# !python render_plan.py --help | grep -A2 caption-backplate

In [8]:
# Verify font discovery (OPTIONAL)
# !python verify_font_discovery.py

In [9]:
# Check font paths (OPTIONAL)
# !python -c "from phase3.typography import FONT_PATHS; print(FONT_PATHS)"
# print("✅ Pre-testingvthe discovery of Amiri fonts completed!")

In [10]:
# Install anthropic
!pip install anthropic --quiet
print("✴️ Anthropic installed!")

✴️ Anthropic installed!


In [11]:
# !pip install arabic-reshaper python-bidi
# print("✅ Arabic_reshaper and python-bidi installed!")

In [12]:
# Colab API Keys
print("Retrieving the Anthropic, Pexels & Hugging Face API keys & token ...")
from google.colab import userdata
import os

# Retrieve the Anthropic API key from Colab Secrets
anthropic_api_key = userdata.get('ANTHROPIC_API_KEY')
pexels_api_key = userdata.get('PEXELS_API_KEY')
hf_token = userdata.get('HF_TOKEN')

# Set it as an environment variable for phase3_run.py to use
if anthropic_api_key:
    os.environ['ANTHROPIC_API_KEY'] = anthropic_api_key
    print("🔑 Anthropic API key loaded.")
else:
    print("❌ Warning: ANTHROPIC_API_KEY not found in Colab Secrets. Please ensure it's set correctly.")

if pexels_api_key:
    os.environ['PEXELS_API_KEY'] = pexels_api_key
    print("🔑 Pexels API key loaded.")
else:
    print("❌ Warning: PEXELS_API_KEY not found in Colab Secrets. Please ensure it's set correctly.")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("🔑 Hugging Face token loaded.")
else:
    print("❌ Warning: HF_TOKEN not found in Colab Secrets. Please ensure it's set correctly.")

Retrieving the Anthropic, Pexels & Hugging Face API keys & token ...
🔑 Anthropic API key loaded.
🔑 Pexels API key loaded.
🔑 Hugging Face token loaded.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Step 1 · Forced alignment  (the real one — auto → WhisperX)
# Every caption's timing comes from this file, so it is computed once
# here, verified by the next cell, and reused by the plan cell.  Takes
# a few minutes on a Colab GPU runtime.
# ════════════════════════════════════════════════════════════════════
!python phase3_run.py \
    --script         {SCRIPT_FILE} \
    --audio          {AUDIO_FILE} \
    --align-only \
    --align-backend  auto \
    --save-alignment {TIMINGS_FILE}

# Smoke test only (instant, timings are estimates — never plan against it):
#   --align-backend interpolated

print("✅ Alignment saved to", TIMINGS_FILE)


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Gate 1 · does the SCRIPT say what the NARRATION says?
# Captions are drawn from the script and timed from the alignment.  If
# the MP3 was synthesised from a different revision of the script, every
# caption in the divergent stretch shows words nobody is speaking — and
# no planner or renderer can repair that.  Fix the DATA and re-run:
# either re-synthesise narration.mp3 from this script, or restore the
# script the MP3 was read from.
# ════════════════════════════════════════════════════════════════════
import subprocess

rc = subprocess.run(["python", "verify_narration.py",
                     "--script",       SCRIPT_FILE,
                     "--word-timings", TIMINGS_FILE]).returncode
if rc != 0:
    raise RuntimeError(
        "Script and narration disagree — stop here.  Rendering now would "
        "burn captions the audience is not hearing.  See the regions listed "
        "above.")
print("✅ Script and narration agree — safe to plan.")


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Step 2 · Shot plan (one Sonnet call, ~$0.10)
# Plans against the timings verified above, so WhisperX runs once per
# session and planner + captions can never disagree about what was
# said or when.
# ════════════════════════════════════════════════════════════════════
# Produces output/shot_plan.json
!python phase3_run.py \
    --script         {SCRIPT_FILE} \
    --audio          {AUDIO_FILE} \
    --book-title     "{BOOK_TITLE}" \
    --character-name "{CHARACTER_NAME}" \
    --plan-only \
    --word-timings   {TIMINGS_FILE} \
    --save-plan      {PLAN_FILE}

print("✅ Regenerating the plan completed!")

In [ ]:
# Structural audit of the plan — shot count, holds, auto-splits.
!python audit_plan.py {PLAN_FILE}
# Expect: ~43 shots, <10% auto-split
# Add --review-dir {REVIEW_DIR} AFTER prebuild to get the effective-holds
# report with dossier-resolved duplicate detection (perceived pacing).

print("✅ Audit completed!")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Gate 2 · what will the audience actually READ?
# audit_captions.py rebuilds the exact caption track render_plan.py will
# burn and walks every narrated word: readable / LOST / DUPLICATED /
# LEAKED / CAPTION-OVER-CARD.  Running it here catches a bad plan in
# seconds instead of after a ~40-minute render.
#
# Reference numbers from the last verified plan: 641 readable, 10 LOST,
# 0 DUPLICATED, 0 LEAKED, 0 CAPTION-OVER-CARD.
# ════════════════════════════════════════════════════════════════════
import subprocess

rc = subprocess.run(["python", "audit_captions.py",
                     "--plan",         PLAN_FILE,
                     "--word-timings", TIMINGS_FILE,
                     "--script",       SCRIPT_FILE,
                     "--max-lost",     str(MAX_LOST_WORDS)]).returncode
if rc != 0:
    raise RuntimeError(
        "Caption audit failed — do not render yet.  DUPLICATED / LEAKED / "
        "CAPTION-OVER-CARD are plan defects; a LOST count well above the "
        "budget means typography cards are covering narration.  Re-run the "
        "plan cell, or report the numbers above before spending a render.")
print("✅ Caption track is clean — safe to build assets and render.")


In [16]:
# Trimming book cover (OPTIONAL)
# !python trim_book_cover.py overrides/book_cover/my_book.jpg

In [17]:
import os
from pathlib import Path

# Ensure the parent directory for OUTPUT_FILE exists
Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

# Construct the command using an f-string for proper variable interpolation
# Note: $ANTHROPIC_API_KEY and $PEXELS_API_KEY are environment variables,
# and are correctly handled by the shell command, so they don't need Python interpolation.
command = f"""
python prebuild_assets.py \
    --plan           \"{PLAN_FILE}\" \
    --script         \"{SCRIPT_FILE}\" \
    --parallax \
    --book-title     \"{BOOK_TITLE}\" \
    --character-name \"{CHARACTER_NAME}\" \
    --anthropic-key  \"$ANTHROPIC_API_KEY\" \
    --pexels-key     \"$PEXELS_API_KEY\" \
    --photo-bank-max-uses {PHOTO_BANK_MAX_USES} \
    --review-dir     \"{REVIEW_DIR}\"
"""

# Execute the command using get_ipython().system() for robustness
get_ipython().system(command)

print("🧩 Prebuild completed!")
print()
print("🎪 Expected log lines (confirm above):")
print("   Portrait pool detected at /content/Lamahat/resources/character — skipping pinned-portrait copy")
print("   Book cover directory pool detected at /content/Lamahat/resources/book_cover — skipping prebuild copy")
print("   Photo bank auto-detected at /content/Lamahat/resources/photo_bank (Path C) — if present:")
print("     photo_bank: N/59 image shots assigned from M bank photos")
print("     (curated photos become each assigned shot's chosen winner;")
print("      covered shots [bank/pool] now SKIP the web waterfall by default;")
print("      add --fetch-covered to capture web alternates for them)")
print("   Image/score cache: … — point $LAMAHAT_CACHE at Drive to persist it")

INFO    phase3.prebuild  Loaded plan: 85 shots
INFO    phase3.prebuild  Image-needing shots: 50 (the rest are typography)
INFO    phase3.prebuild  Portrait pool detected at /content/Lamahat/resources/character — skipping pinned-portrait copy (pool will be used at render time)
INFO    phase3.prebuild  Book cover directory pool detected at /content/Lamahat/resources/book_cover — skipping prebuild copy (render_plan will auto-discover it)
INFO    phase3.prebuild  Photo bank auto-detected at /content/Lamahat/resources/photo_bank
INFO    phase3.prebuild  Photo bank: captioning 48 photo(s) (cached ones are free)…
INFO    phase3.sources.photo_bank  photo_bank: captioned abdulhamid_ii.jpg — Portrait of Ottoman Sultan Abdulhamid II in formal military dress unif
INFO    phase3.sources.photo_bank  photo_bank: captioned al-askari_funeral.jpg — Ceremonial procession with official in military regalia and automobile
INFO    phase3.sources.photo_bank  photo_bank: captioned arab_ambitions.jpg — Jafar al

In [ ]:
# Zip prebuild assets
#
# Make the dossier SELF-CONTAINED first.  prebuild_assets.py only writes
# the per-shot candidate folders; the plan, the word timings and the
# script live beside it in output/.  Without them the render-only route
# has no way to re-derive a caption track, and audit_captions.py /
# regenerate_captions.py cannot run against the dossier at all.
import shutil
from pathlib import Path as _Path

for _name in (PLAN_FILE, TIMINGS_FILE, SCRIPT_FILE):
    _src = _Path(_name)
    if not _src.exists():
        print(f"  ⚠️ missing {_src} — dossier will be incomplete")
        continue
    _dst = _Path(REVIEW_DIR) / ("script.txt" if _src == _Path(SCRIPT_FILE)
                                else _src.name)
    if _src.resolve() != _dst.resolve():
        shutil.copy2(_src, _dst)
    print(f"  ✓ {_dst}")

import os
import zipfile

# Cap the dossier at the top-3 candidates per shot before zipping.
# New prebuilds already write top-3 only (this is a no-op for them);
# for dossiers from older prebuilds it trims the bulk.
from pathlib import Path as _P
from phase3 import slim_review_dir
_slim = slim_review_dir(_P(REVIEW_DIR), mode="top", keep_n=3)
print(f"🧹 Dossier slimmed: kept {_slim['kept']}, removed {_slim['removed']} "
      f"({_slim['bytes_removed']/1e6:.1f} MB freed)")

output_dir = REVIEW_DIR
zip_filename = REVIEW_FILE

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to 'output_dir'
            # Skip .mp3 files if any exist (though unlikely in review folder)
            if not file_path.endswith('.mp3'):
                zipf.write(file_path, os.path.relpath(file_path, output_dir))
                files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing: ")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print("⚠️ No files found to zip in the "+REVIEW_DIR+" directory!")

In [19]:
# Standalone test runner
# !python verify_user_marked.py

In [20]:
# !python verify_title_card.py --book-cover overrides/book_cover/my_book.jpg

In [21]:
# !python diagnose_issue4.py --review-dir output/review/   # is everything in place?

In [22]:
# Condition assets
!python condition_assets.py --review-dir {REVIEW_DIR} # --sr realesrgan]; --dry-run

INFO  condition_assets  Shot 2 contain    sr 935x1200 -> 1247x1600 [sr]
INFO  condition_assets  Shot 4 cover_crop crop 2560x1441 -> 2560x1440 [ok]  crop=(0, 1, 2560, 1441)
INFO  condition_assets  Shot 6 cover_crop crop 2560x1441 -> 2560x1440 [ok]  crop=(0, 0, 2560, 1440)
INFO  condition_assets  Shot 7 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 9 toned (documentary palette, source=pexels)
INFO  condition_assets  Shot 9 cover_crop upscale 1880x1253 -> 2560x1440 [upscaled]  crop=(0, 0, 1880, 1058)
INFO  condition_assets  Shot 11 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 13 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 14 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 15 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 17 cover      asis 2560x1440 -> 2560x1440 [ok]
INFO  condition_assets  Shot 18 cover_crop crop 2560x1441 -> 2560x1440 [ok]  crop=(0, 0, 2

In [23]:
# OPTIONAL
# Zip conditioned assets
import os
import zipfile

output_dir = REVIEW_DIR
zip_filename = CONDITIONED_ZIP_FILE

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to 'output_dir'
            # Skip .mp3 files if any exist (though unlikely in review folder)
            if not file_path.endswith('.mp3'):
                zipf.write(file_path, os.path.relpath(file_path, output_dir))
                files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing: ")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print("⚠️ No files found to zip in the", REVIEW_DIR, "directory!")

🤐 Successfully created 'output/conditioned.zip' containing: 
  - output/review/README.txt
  - output/review/photo_bank_assignment_raw.txt
  - output/review/decisions.json
  - output/review/shot_32_portrait/context.txt
  - output/review/shot_32_portrait/bank_jafar_with_colleagues.jpg
  - output/review/shot_32_portrait/candidates.json
  - output/review/shot_56_broll/context.txt
  - output/review/shot_56_broll/wikimedia_a.jpg
  - output/review/shot_56_broll/candidates.json
  - output/review/shot_56_broll/wikipedia_a.jpg
  - output/review/shot_21_broll/wikipedia_b.jpg
  - output/review/shot_21_broll/context.txt
  - output/review/shot_21_broll/wikimedia_a.jpg
  - output/review/shot_21_broll/candidates.json
  - output/review/shot_21_broll/wikipedia_a.jpg
  - output/review/shot_28_broll/pexels_b.jpg
  - output/review/shot_28_broll/context.txt
  - output/review/shot_28_broll/pexels_a.jpg
  - output/review/shot_28_broll/candidates.json
  - output/review/shot_28_broll/wikipedia_a*.jpg
  - output

In [ ]:
import os
from pathlib import Path

# Ensure the parent directory for OUTPUT_FILE exists
Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

# Construct the command parts conditionally in Python
grade_map_arg   = f"--grade-map {GRADE_MAP}" if GRADE_MAP else ""
word_reveal_arg = "--word-reveal" if WORD_REVEAL else ""
# Blank look settings fall through to the pipeline defaults rather than
# being passed as empty strings (argparse would take "" literally).
title_color_arg   = f'--title-color "{TITLE_COLOR}"' if TITLE_COLOR else ""
caption_color_arg = f'--caption-color "{CAPTION_COLOR}"' if CAPTION_COLOR else ""
caption_pos_arg   = f"--caption-pos {CAPTION_POS}" if CAPTION_POS else ""

# Construct the full command string
render_command = f"""
python render_plan.py \
    --plan              "{PLAN_FILE}" \
    --audio             "{AUDIO_FILE}" \
    --music             "{MUSIC_BED}" \
    --music-gain        {str(MUSIC_DB)} \
    --review-dir        "{REVIEW_DIR}" \
    --book-cover-pick   {str(BOOK_COVER_PICK)} \
    --book-cover-fit    {BOOK_COVER_FIT} \
    --book-cover-align  {BOOK_COVER_ALIGN} \
    --typography-family {TYPOGRAPHY_FAMILY} \
    --parallax \
    --typography-over-image \
    --grade             {GRADE} \
    {grade_map_arg} \
    --text-scrim        {TEXT_SCRIM} \
    --overlay-anchor    {OVERLAY_ANCHOR} \
    --title-subtitle    "{TITLE_SUBTITLE}" \
    {word_reveal_arg} \
    --caption-backplate {CAPTION_BACKPLATE} \
    --caption-size      {CAPTION_SIZE} \
    --title-size        {TITLE_SIZE} \
    {caption_color_arg} \
    {caption_pos_arg} \
    {title_color_arg} \
    --output            {OUTPUT_FILE} \
    > {LOG_FILE} 2>&1 &
"""

# Execute the command using get_ipython().system()
get_ipython().system(render_command)

print("Rendering begins ...")
print(f"  log:    {LOG_FILE}")
print(f"  output: {OUTPUT_FILE}")
print()
print("Run the next cell to tail the log until completion.")
print("Rendering could nearly take up to 40 minutes!")

In [25]:
# Monitor rendering progress
import time
from IPython.display import clear_output

log_path = LOG_FILE

print("Monitoring rendering progress...")
while True:
    try:
        # Read the log file contents
        try:
            with open(log_path, "r") as f:
                log_content = f.read()
        except FileNotFoundError:
            log_content = ""

        # Clear cell output and show the last 20 lines
        clear_output(wait=True)
        lines = log_content.splitlines()
        print("\n".join(lines[-20:]))

        # Check if the script's success signature is in the log
        if "Done in" in log_content or "Rendered video →" in log_content:
            print("\n✅ Rendering process completed successfully! Stopped monitoring.")
            break

        time.sleep(5)

    except KeyboardInterrupt:
        print("\n⚠️ Monitoring stopped manually. The script may still be running.")
        break


INFO  phase3.sources.decisions  Shot 80: chosen-file hit bank_present_&_past.jpg
INFO  phase3.render  [render 72%] shot 81/85: typography
INFO  phase3.sources.decisions  Shot 82: portrait pool hit memoirs_in_shelf.jpg (rank 10 of 11)
INFO  phase3.sources  Shot 82: review-dossier hit memoirs_in_shelf.jpg
INFO  phase3.render  Shot 82: using fetched image from review_dossier
INFO  phase3.render  [render 73%] shot 82/85: portrait
INFO  phase3.render  [render 73%] shot 83/85: typography
INFO  phase3.sources.decisions  Shot 82: portrait pool hit memoirs_in_shelf.jpg (rank 10 of 11)
INFO  phase3.render  [render 74%] shot 84/85: typography
INFO  phase3.sources.decisions  Shot 82: portrait pool hit memoirs_in_shelf.jpg (rank 10 of 11)
INFO  phase3.render  [render 75%] shot 85/85: typography
INFO  phase3.render  [render 80%] concat all shots
INFO  phase3.render  [render 86%] generating captions
INFO  phase3.render  [render 92%] mux audio and captions
INFO  phase3.render  Color grade applied: war

In [26]:
# !python diagnose_grade.py

In [27]:
# !python diagnose_captions.py --plan output/shot_plan.json   # inspect actual ASS events

In [28]:
# Zip output files for exporting
import os
import zipfile

output_dir = OUTPUT_BASE_DIR
zip_filename = FINAL_ZIP_FILE

# Get all files in the output directory
files_to_zip = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))]

# Filter out the .mp3 file
filtered_files = [f for f in files_to_zip if not f.endswith('.mp3')]

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in filtered_files:
        # Add file to zip, preserving directory structure relative to 'output_dir'
        zipf.write(file_path, os.path.relpath(file_path, output_dir))

print(f"🤐 Successfully created '{zip_filename}' containing: ")
for f in filtered_files:
    print(f"  - {f}")

🤐 Successfully created 'output_files.zip' containing: 
  - output/review.zip
  - output/render.log
  - output/word_timings.json
  - output/shot_plan.json
  - output/conditioned.zip
  - output/planner_raw_response.txt
  - output/final_cut_B.mp4


In [30]:
# Save zipped file to Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Define the destination directory and file path
dest_dir = DRIVE_SAVE_DIR
dest_file_path = os.path.join(dest_dir, FINAL_ZIP_FILE)

# Create the destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)
print(f"Destination directory '{dest_dir}' ensured to exist.")

# --- Test write access to the directory ---
test_file = os.path.join(dest_dir, 'test_write.txt')
try:
    with open(test_file, 'w') as f:
        f.write('This is a test file.\n')
    print(f"✅ Successfully wrote test file to '{test_file}'.")
    os.remove(test_file) # Clean up the test file
    print(f"Test file '{test_file}' removed.")
except Exception as e:
    print(f"Error writing test file to '{test_file}': {e}")
    print("⚠️ It seems there might be a permissions or access issue with Google Drive.")
    # Exit or raise an error if write access fails
    raise
# ----------------------------------------

shutil.copy(FINAL_ZIP_FILE, dest_file_path)
print(f"📽️ Files are saved to Google Drive at '{dest_file_path}'.")

Mounted at /content/drive
Destination directory '/content/drive/MyDrive/_Phase3/output' ensured to exist.
✅ Successfully wrote test file to '/content/drive/MyDrive/_Phase3/output/test_write.txt'.
Test file '/content/drive/MyDrive/_Phase3/output/test_write.txt' removed.
📽️ Files are saved to Google Drive at '/content/drive/MyDrive/_Phase3/output/output_files.zip'.


In [31]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile
from pathlib import Path

SRC = {
    'main': '/content/drive/MyDrive/_Phase3/output/output_files.zip',
    'ro':   '/content/drive/MyDrive/_Phase3/output/ro/output_files_ro.zip',
    'user': '/content/drive/MyDrive/_Phase3/sources/Archive.zip',
}
DEST = Path('/content/drive/MyDrive/_Phase3/audit'); DEST.mkdir(exist_ok=True)
KEEP = ('.json', '.txt', '.log', '.md', '.ass')

for tag, zp in SRC.items():
    zp = Path(zp)
    if not zp.exists():
        print('MISSING:', zp); continue
    out = DEST / tag; out.mkdir(exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        names = z.namelist()
        # full manifest (so the audit sees WHICH shot folders/images exist)
        (DEST / f'{tag}_manifest.txt').write_text('\n'.join(names))
        for m in names:
            if m.endswith(KEEP) and not m.endswith('/'):
                (out / m.replace('/', '__')).write_bytes(z.read(m))
    print(tag, '→ done,', len(names), 'entries')
print('All done — tell Claude to read _Phase3/audit/')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
main → done, 7 entries
MISSING: /content/drive/MyDrive/_Phase3/output/ro/output_files_ro.zip
user → done, 611 entries
All done — tell Claude to read _Phase3/audit/
